# 02 - Baseline Models And Submission

This notebook builds reproducible starter baselines.

Modeling choices here are deliberately conservative:
- exclude `ticker` from the first baseline
- use only features available at the observation date
- evaluate on a 2022 holdout before fitting on all training rows

The goal is not leaderboard magic. The goal is a clean baseline that the team can improve.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['period_start', 'period_end'])
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

train.shape, test.shape

((23070, 39), (8520, 36))

In [2]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame.copy()

    features['obs_year'] = features['start_year']

    if 'period_start' in features.columns:
        features['obs_quarter'] = features['period_start'].dt.quarter
        features['obs_month'] = features['period_start'].dt.month

    drop_columns = ['id', 'ticker', 'start_year', 'period_start', 'period_end', 'return_pct']
    existing_drop_columns = [col for col in drop_columns if col in features.columns]
    features = features.drop(columns=existing_drop_columns)

    numeric_columns = features.select_dtypes(include=[np.number]).columns.tolist()
    return features[numeric_columns]


X_full = build_features(train)
X_test = build_features(test)
y_full = train['return_pct']

print('Number of modeling features:', X_full.shape[1])
display(X_full.head())

Number of modeling features: 36


,pe_ttm,price_to_book,price_to_sales,growth_pe_ratio,gross_margin,operating_margin,net_margin,roa,roe,rote,revenue_growth_3y,revenue_growth_yoy,revenue_ttm,net_income_ttm,income_before_tax,eps_basic,eps_diluted,total_assets,stockholders_equity,current_assets,current_liabilities,long_term_debt,goodwill,inventory,current_ratio,quick_ratio,debt_to_equity,dividend_yield,dividends_ttm,dividends_paid_ttm,shares_outstanding,shares_diluted,sector_code,obs_year,obs_quarter,obs_month
0,25.28,38.23,6.68,1.95,43.75,30.82,26.41,29.07,151.24,151.24,49.34,18.63,3.860170e+11,1.019350e+11,3.013900e+10,1.54,1.52,3.506620e+11,6.739900e+10,1.181800e+11,1.275080e+11,2.066460e+11,0.0,5.460000e+09,0.93,0.88,3.07,0.57,1.473400e+10,1.468700e+10,1.620757e+10,1.640332e+10,0.0,2022,1,3
1,20.97,35.95,5.39,2.37,43.26,27.82,25.71,29.63,171.46,171.46,49.61,11.63,3.875420e+11,9.963300e+10,2.306600e+10,1.20,1.20,3.363090e+11,5.810700e+10,1.122920e+11,1.298730e+11,1.894000e+11,0.0,5.433000e+09,0.86,0.82,3.26,0.71,1.477800e+10,1.473400e+10,1.609538e+10,1.626220e+10,0.0,2022,2,6
2,22.23,43.78,5.63,2.32,43.31,30.29,25.31,28.29,196.96,196.96,51.56,7.79,3.943280e+11,9.980300e+10,1.191030e+11,6.15,6.11,3.527550e+11,5.067200e+10,1.354050e+11,1.539820e+11,1.979180e+11,0.0,4.946000e+09,0.88,0.85,3.91,0.67,1.484100e+10,1.479300e+10,1.594342e+10,1.632582e+10,0.0,2022,3,9
3,20.13,33.78,4.94,2.22,42.96,30.74,24.56,27.45,167.77,167.77,44.77,2.44,3.875370e+11,9.517100e+10,3.562300e+10,1.89,1.88,3.467470e+11,5.672700e+10,1.287770e+11,1.372860e+11,1.992540e+11,0.0,6.820000e+09,0.94,0.89,3.51,0.78,1.487700e+10,1.484000e+10,1.584241e+10,1.595572e+10,0.0,2022,4,12
4,23.43,25.84,5.49,1.35,42.51,30.70,23.45,22.63,110.31,110.31,31.52,21.43,3.254060e+11,7.631100e+10,2.801100e+10,1.41,1.40,3.371580e+11,6.917800e+10,1.214650e+11,1.063850e+11,2.172840e+11,0.0,5.219000e+09,1.14,1.09,3.14,0.80,1.422700e+10,1.421200e+10,1.668630e+10,1.692916e+10,0.0,2021,1,3


In [3]:
train_mask = train['start_year'] < 2022
valid_mask = ~train_mask

X_train = X_full.loc[train_mask]
y_train = y_full.loc[train_mask]
X_valid = X_full.loc[valid_mask]
y_valid = y_full.loc[valid_mask]

print('X_train shape:', X_train.shape)
print('X_valid shape:', X_valid.shape)

X_train shape: (16436, 36)
X_valid shape: (6634, 36)


In [4]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

ridge_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        (
            'model',
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_depth=4,
                max_iter=300,
                min_samples_leaf=40,
                random_state=42,
            ),
        ),
    ]
)

models = {
    'dummy_mean': DummyRegressor(strategy='mean'),
    'dummy_median': DummyRegressor(strategy='median'),
    'ridge': ridge_pipeline,
    'hist_gradient_boosting': hgb_pipeline,
}

scores = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    score = rmse(y_valid, preds)
    scores.append({'model': name, 'rmse': score})
    fitted_models[name] = model

scores = pd.DataFrame(scores).sort_values('rmse')
display(scores)

,model,rmse
1,dummy_median,65.350859
0,dummy_mean,65.390587
3,hist_gradient_boosting,65.942368
2,ridge,95.594241


In [5]:
best_model_name = scores.iloc[0]['model']
best_model = fitted_models[best_model_name]

valid_predictions = best_model.predict(X_valid)
residuals = pd.DataFrame(
    {
        'ticker': train.loc[valid_mask, 'ticker'].values,
        'start_year': train.loc[valid_mask, 'start_year'].values,
        'actual': y_valid.values,
        'predicted': valid_predictions,
    }
)
residuals['abs_error'] = (residuals['actual'] - residuals['predicted']).abs()

print('Best validation model:', best_model_name)
display(residuals.sort_values('abs_error', ascending=False).head(15))

Best validation model: dummy_median


,ticker,start_year,actual,predicted,abs_error
1721,SLNO,2022,1932.83,3.645,1929.185
1720,SLNO,2022,1667.07,3.645,1663.425
6287,ACIC,2022,1050.00,3.645,1046.355
1198,CVNA,2022,1016.88,3.645,1013.235
5139,AAOI,2022,922.22,3.645,918.575
6288,ACIC,2022,792.45,3.645,788.805
4270,CIFR,2022,637.50,3.645,633.855
3178,IMVT,2022,587.99,3.645,584.345
2170,MARA,2022,586.84,3.645,583.195
5631,AVDL,2022,575.41,3.645,571.765


## Refit On All Training Data

After selecting a baseline model using the holdout, refit it on the full training set before generating the Kaggle submission.

In [6]:
final_model = models[best_model_name]
final_model.fit(X_full, y_full)
test_predictions = final_model.predict(X_test)

submission = sample_submission.copy()
submission['return_pct'] = test_predictions

submission_path = SUBMISSION_DIR / f'{best_model_name}_baseline.csv'
submission.to_csv(submission_path, index=False)

print('Saved submission to:', submission_path)
display(submission.head())
print('Submission rows:', len(submission))

Saved submission to: /home/leonid/Repositories/AIC/kaggle-competition-stock-return-fundamentals/submissions/dummy_median_baseline.csv


,id,return_pct
0,0,3.5
1,1,3.5
2,2,3.5
3,3,3.5
4,4,3.5


Submission rows: 8520


## Next Experiments

Once this notebook runs cleanly, the next sensible experiments are:

- target clipping or winsorization on the training fold only
- sector-relative features
- log transforms for scale-heavy accounting variables
- LightGBM or XGBoost with the same time-based validation split
- model blending once two models show independent value